In [ ]:
# ==============================================================================
# Cell 1: Notebook Setup
# ==============================================================================
# --- Future Imports (must be first) ---
from __future__ import annotations

# --- Standard Library Imports ---
import glob
import os
import subprocess
import sys
import time
import warnings
from pathlib import Path

# --- Notebook Magic Commands ---
%load_ext autoreload
%autoreload 2

# --- Suppress Warnings & Configure Environment for Cleaner Output ---
# Set environment variables before importing TensorFlow
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # Disable GPU usage
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # Suppress TensorFlow C++ logs
os.environ.setdefault("ABSL_LOG_LEVEL", "3")         # Suppress Abseil logs

# Filter Python warnings specifically from tensorflow_addons
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow_addons")

# Attempt to import TensorFlow and set its logger level
try:
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")
except ImportError:
    print("TensorFlow not found, skipping its configuration.")
    pass

# --- Add Project Source to Python Path ---
# This allows importing local modules like 'forecast_pipeline' and 'hpo'
project_root = Path.cwd().parents[1]
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    print(f"Added '{src_path}' to Python path.")

# --- Local Application Imports & Configuration ---
from forecast_pipeline.io_utils import configure_logging
from hpo.posthoc_filtering import print_campaign_summary

# Configure logging for the notebook session
configure_logging()

print("✅ Setup complete.")

In [ ]:
# ==============================================================================
# CONTROL PANEL (Centralized Configuration)  <-- keep this block at the top
# ==============================================================================
CAMPAIGN_GROUP = "HPO_115_Lag_100_Horizon_150"       # e.g., "DEMO_1" or None

FAMILIES_TO_RUN = [ "arps", "seq2", "darts"]       # The order here defines the execution order
FAMILIES_TO_RUN = [ "arps"]       # The order here defines the execution order

MAX_CONCURRENT_RUNS_BY_FAMILY = {
    "seq2": 8,
    "darts": 3,
    "arps": 3,
}
NAME_CONTAINS = []                       # Optional filename filters, e.g., ["UNISIM_IV_P12", "Seq2PIN"]
DRY_RUN = False

# Priority for running jobs *within* a family batch (lower runs first)
ARCHITECTURE_PRIORITY = {
    "seq2context": 1,
    "seq2trend":   2,
    "seq2pin":     3,
    "darts":       4,
    "arps":        5,
}

DATASET_PRIORITY = {
    "volve": 0,
    "unisim_iv": 1,
    "unisim": 2,
    # adicione outros apelidos/prefixos conforme seus arquivos .yaml
}
DEFAULT_DATASET_PRIORITY = 99  # se não casar com nada acima

# ==============================================================================
# TIMING UTILITIES (plug-and-play)
# ==============================================================================
from typing import List, Dict, Optional
from pathlib import Path
import time
import subprocess
import glob

def _fmt_hhmm(seconds: float) -> str:
    """Format seconds as HhMm (zero-padded)."""
    seconds = max(0, int(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    return f"{h:02d}h{m:02d}m"

# ==============================================================================
# HELPER FUNCTIONS (with timing)
# ==============================================================================
def filter_and_sort_campaigns(files: List[Path], name_filters: List[str]) -> List[Path]:
    """Aplica filtros e ordena campanhas por prioridade de dataset e arquitetura."""
    def _dataset_priority(p: Path) -> int:
        s = p.stem.lower()
        for k, prio in DATASET_PRIORITY.items():
            if k in s:
                return prio
        return DEFAULT_DATASET_PRIORITY

    def _arch_priority(p: Path) -> int:
        s = p.stem.lower()
        for arch_key, prio in ARCHITECTURE_PRIORITY.items():
            if arch_key in s:
                return prio
        return 99  # prioridade baixa se não casar

    def _name_ok(p: Path) -> bool:
        if not name_filters:
            return True
        s = p.name.lower()
        return any(f.lower() in s for f in name_filters)

    # Ordena por (dataset -> arquitetura -> nome) para ter ordem determinística
    return sorted(
        [p for p in files if _name_ok(p) and not p.stem.startswith("validation")],
        key=lambda p: (_dataset_priority(p), _arch_priority(p), p.name.lower()),
    )
    
def discover_campaigns_for_family(family_slug: str, hpo_root: Path, group: Optional[str]) -> List[Path]:
    """Finds all YAML files for a specific family."""
    search_dir = hpo_root / group / family_slug if group else hpo_root / family_slug
    return [Path(p) for p in glob.glob(str(search_dir / "*.yaml"))]


def run_batch(command_queue: List[Dict], max_concurrent: int, family_name: str, metrics_store: Dict) -> None:
    """
    Launches and monitors a single batch of subprocesses with timing.
    - Records per-campaign elapsed time (launch -> finished).
    - Records family wall-clock (batch start -> batch end).
    - Aggregates sum of campaign times for the family.
    """
    active_processes = []
    completed_count = 0
    total_jobs = len(command_queue)

    # Metrics containers
    family_key = family_name.lower()
    metrics_store.setdefault(family_key, {
        "campaigns": [],                 # list of {"name", "elapsed_sec", "status", "log_path"}
        "batch_wall_sec": 0.0,          # wall-clock of the whole family batch
        "sum_campaigns_sec": 0.0        # sum of all campaign times (even if parallel)
    })

    if total_jobs == 0:
        print(f"  -> No jobs to run for {family_name.upper()}. Skipping.")
        return

    batch_start = time.perf_counter()
    print(f"\n🚀 Starting {family_name.upper()} batch ({total_jobs} jobs, {max_concurrent} parallel)...")
    try:
        while completed_count < total_jobs:
            # Fill available slots with new jobs
            while len(active_processes) < max_concurrent and command_queue:
                job = command_queue.pop(0)
                print(f"  -> Launching '{job['name']}'...")
                log_handle = open(job["log_path"], "w", encoding="utf-8")
                proc = subprocess.Popen(job["command"], stdout=log_handle, stderr=subprocess.STDOUT)
                job_runtime = {
                    "name": job["name"],
                    "start_sec": time.perf_counter(),
                    "end_sec": None,
                    "elapsed_sec": None,
                    "status": "RUNNING",
                    "log_path": str(job["log_path"])
                }
                active_processes.append({"process": proc, "log_file": log_handle, "name": job["name"], "runtime": job_runtime})

            # Poll running jobs for completion
            for i, info in reversed(list(enumerate(active_processes))):
                if info["process"].poll() is not None:
                    completed_count += 1
                    if not info["log_file"].closed:
                        info["log_file"].close()

                    end = time.perf_counter()
                    runtime = info["runtime"]
                    runtime["end_sec"] = end
                    runtime["elapsed_sec"] = end - runtime["start_sec"]
                    succeeded = (info["process"].returncode == 0)
                    runtime["status"] = "SUCCESS" if succeeded else f"FAILED (code {info['process'].returncode})"

                    # Persist metrics
                    metrics_store[family_key]["campaigns"].append(runtime)
                    metrics_store[family_key]["sum_campaigns_sec"] += runtime["elapsed_sec"]

                    print(f"  -> ✅ '{info['name']}' finished. {runtime['status']}. "
                          f"Elapsed: {_fmt_hhmm(runtime['elapsed_sec'])}. "
                          f"({completed_count}/{total_jobs})")
                    active_processes.pop(i)

            time.sleep(6)  # Avoid busy-waiting
    finally:
        # Graceful shutdown for any remaining active processes in this batch
        if active_processes:
            print(f"\n--- SHUTDOWN SEQUENCE FOR {family_name.upper()} BATCH ---")
            for info in active_processes:
                print(f"  -> Terminating '{info['name']}' (PID: {info['process'].pid})")
                info["process"].terminate()
                try:
                    info["process"].wait(timeout=15)
                except subprocess.TimeoutExpired:
                    print(f"     '{info['name']}' unresponsive, force killing...")
                    info["process"].kill()
                finally:
                    if not info["log_file"].closed:
                        info["log_file"].close()
            print(f"✅ {family_name.upper()} batch shutdown complete.")

    batch_end = time.perf_counter()
    metrics_store[family_key]["batch_wall_sec"] = batch_end - batch_start

    status_message = (
        "🎉 All campaigns completed!"
        if completed_count == total_jobs
        else f"⏹️ Batch interrupted. {completed_count}/{total_jobs} completed."
    )
    print(f"\n{status_message} (Family: {family_name.upper()})")
    print(f"   ↳ Family wall-clock: {_fmt_hhmm(metrics_store[family_key]['batch_wall_sec'])}")
    print(f"   ↳ Sum of campaign times: {_fmt_hhmm(metrics_store[family_key]['sum_campaigns_sec'])}")

# ==============================================================================
# MAIN ORCHESTRATOR (unchanged flow + timing summary)
# ==============================================================================
HPO_ROOT = src_path / "experiment_configs" / "hpo_campaigns"
run_script_path = src_path / "run_campaign.py"
logs_dir = Path.cwd() / "logs"
logs_dir.mkdir(exist_ok=True)

# Build job queue per family
all_family_jobs: Dict[str, List[Dict]] = {}
for family in FAMILIES_TO_RUN:
    family_files = discover_campaigns_for_family(family, HPO_ROOT, CAMPAIGN_GROUP)
    sorted_files = filter_and_sort_campaigns(family_files, NAME_CONTAINS)

    command_queue = []
    for p in sorted_files:
        log_path = logs_dir / f"{p.stem}.log"
        cmd = ["python", str(run_script_path), str(p.resolve())]
        command_queue.append({"command": cmd, "log_path": log_path, "name": p.stem})
    all_family_jobs[family] = command_queue

# Execution plan (unchanged)
print(f"📦 Project root: {project_root}")
print(f"📁 src:          {src_path}")
print(f"🎯 Group:        {CAMPAIGN_GROUP}")
print(f"🏷️ Families to run (in order): {FAMILIES_TO_RUN}")

total_found = sum(len(jobs) for jobs in all_family_jobs.values())
if not total_found:
    print("\n⚠️ No campaigns found for any family. Adjust filters and re-run this cell.")
else:
    for family, jobs in all_family_jobs.items():
        max_runs = MAX_CONCURRENT_RUNS_BY_FAMILY.get(family, 1)
        job_info = [{"name": job["name"], "config_path": str(job["command"][-1])} for job in jobs]
        if job_info:
            print_campaign_summary(job_info, max_runs)
        else:
            print(f"\nNo campaigns found for family: {family.upper()}")

if DRY_RUN or not total_found:
    print("\nDRY RUN enabled or no jobs found. Exiting.")
    raise SystemExit("Exiting due to DRY_RUN or no jobs found.")

# --- Execute sequential batches with timing and final summary ---
_global_start = time.perf_counter()
_family_metrics: Dict[str, Dict] = {}

try:
    for family in FAMILIES_TO_RUN:
        command_queue = all_family_jobs.get(family, [])
        max_concurrent_for_family = MAX_CONCURRENT_RUNS_BY_FAMILY.get(family, 1)  # Default 1
        run_batch(command_queue, max_concurrent_for_family, family, _family_metrics)
except KeyboardInterrupt:
    print("\n\n🛑 KeyboardInterrupt detected. Initiating global shutdown...")
finally:
    _global_end = time.perf_counter()
    print("\n--- GLOBAL EXECUTION FINISHED ---")
    print(f"Total wall-clock: {_fmt_hhmm(_global_end - _global_start)}")

    # Pretty family summary
    if _family_metrics:
        print("\n===== FAMILY RUNTIME SUMMARY =====")
        for fam, data in _family_metrics.items():
            print(f"\n🔹 Family: {fam.upper()}")
            print(f"   • Family wall-clock: {_fmt_hhmm(data['batch_wall_sec'])}")
            print(f"   • Sum of campaign times: {_fmt_hhmm(data['sum_campaigns_sec'])}")
            if data["campaigns"]:
                print(f"   • Campaigns ({len(data['campaigns'])}):")
                # Sort campaigns by elapsed descending to surface long runners
                for c in sorted(data["campaigns"], key=lambda x: x["elapsed_sec"], reverse=True):
                    print(f"       - {c['name']}: {_fmt_hhmm(c['elapsed_sec'])}  [{c['status']}]  (log: {c['log_path']})")


In [ ]:
# import os
# import re
# from lxml import etree
# from dataclasses import dataclass
# from typing import Dict, Optional, Any, Tuple, Set

# # ==============================================================================
# # 0) Helpers
# # ==============================================================================
# SVG_NS = "http://www.w3.org/2000/svg"
# NS = {"svg": SVG_NS}

# URL_REF_RE = re.compile(r"url\(\s*#([^)]+)\s*\)")
# STYLE_KV_RE = re.compile(r"([\w-]+)\s*:\s*([^;]+)")
# LEN_RE = re.compile(r"^\s*([+-]?\d*\.?\d+)\s*([a-zA-Z%]*)\s*$")

# def qn(tag: str) -> str:
#     return f"{{{SVG_NS}}}{tag}"

# def local_tag(tag: str) -> str:
#     if not isinstance(tag, str):
#         return ""
#     return tag.split("}", 1)[1] if tag.startswith("{") else tag

# def parse_len(v: str) -> Tuple[Optional[float], str]:
#     """Parse '12px' -> (12.0,'px'); '10' -> (10.0,''); None -> (None,'')"""
#     if not v:
#         return None, ""
#     m = LEN_RE.match(v)
#     if not m:
#         return None, ""
#     return float(m.group(1)), (m.group(2) or "").strip()

# def fmt_len(num: float, unit: str, nd: int = 3) -> str:
#     s = f"{num:.{nd}f}".rstrip("0").rstrip(".")
#     return s + (unit or "")

# def parse_style(style_str: str) -> Dict[str, str]:
#     if not style_str:
#         return {}
#     return dict(STYLE_KV_RE.findall(style_str))

# def build_style(d: Dict[str, str]) -> str:
#     keys = sorted(d.keys())
#     return ";".join([f"{k}:{d[k]}" for k in keys if d[k] is not None and str(d[k]).strip() != ""])

# def replace_url_refs_in_text(text: str, old_id: str, new_id: str) -> str:
#     """Replace url(#old) -> url(#new) in attributes/styles."""
#     if not text:
#         return text
#     return re.sub(r"url\(\s*#%s\s*\)" % re.escape(old_id), f"url(#{new_id})", text)

# # ==============================================================================
# # 1) Theme + Data (o seu mesmo)
# # ==============================================================================
# @dataclass
# class Theme:
#     name: str
#     colors: Dict[str, str]
#     knobs: Dict[str, Any]
#     font_family: str

# THEMES_DATA = {
#     "NATURE": Theme(
#         name="Nature / Scientific Reports",
#         colors={
#             "primary": "#34495e", "danger": "#c0392b", "success": "#27ae60",
#             "neutral": "#7f8c8d", "accent": "#f39c12", "bg_panel": "#f8f9fa",
#             "stroke_grid": "#ecf0f1"
#         },
#         knobs={"stroke_scale": 1.2, "corner_radius": 6, "shadow_opacity": 0.15, "font_scale": 1.0},
#         font_family="'Inter', 'Helvetica Neue', Arial, sans-serif"
#     ),
#     "ELSEVIER": Theme(
#         name="Elsevier / SPE Standard",
#         colors={
#             "primary": "#0073e6", "danger": "#d63031", "success": "#00b894",
#             "neutral": "#636e72", "accent": "#e17055", "bg_panel": "#ffffff",
#             "stroke_grid": "#dfe6e9"
#         },
#         knobs={"stroke_scale": 1.0, "corner_radius": 0, "shadow_opacity": 0.0, "font_scale": 1.1},
#         font_family="'Arial', sans-serif"
#     ),
#     "DARK_MODERN": Theme(
#         name="Dark Mode (Presentation)",
#         colors={
#             "primary": "#54a0ff", "danger": "#ff6b6b", "success": "#1dd1a1",
#             "neutral": "#c8d6e5", "accent": "#feca57", "bg_panel": "#222f3e",
#             "stroke_grid": "#576574"
#         },
#         knobs={"stroke_scale": 1.5, "corner_radius": 12, "shadow_opacity": 0.5, "font_scale": 1.2},
#         font_family="'Roboto', sans-serif"
#     ),
#     "GEOLOGY": Theme(
#         name="Geology / Earth Tones",
#         colors={
#             "primary": "#5D4037", "danger": "#D32F2F", "success": "#388E3C",
#             "neutral": "#616161", "accent": "#FBC02D", "bg_panel": "#EFEBE9",
#             "stroke_grid": "#D7CCC8"
#         },
#         knobs={"stroke_scale": 0.9, "corner_radius": 2, "shadow_opacity": 0.1, "font_scale": 1.0},
#         font_family="'Georgia', serif"
#     )
# }

# ORIGINAL_MAP = {
#     "primary": ["#206a92", "#206A92", "#1d5b7d"],
#     "danger":  ["#b22222", "#B22222", "#8b0000"],
#     "success": ["#1e4e2d", "#1e5631", "#1E5631"],
#     "neutral": ["#a9a9a9", "#757575", "#2e2e2e", "#6c757d"],
#     "accent":  ["#e3c800", "#B8860B", "#f1c40f"],
#     "bg_white": ["#ffffff", "#FFFFFF"]
# }

# # ==============================================================================
# # 2) Options — “não distorcer” por default
# # ==============================================================================
# @dataclass
# class Options:
#     # Inkscape repair passes
#     dedupe_duplicate_ids: bool = True
#     drop_broken_url_refs: bool = True

#     # Styling
#     apply_shadow: bool = False            # OFF por default (você reportou distorção)
#     set_font_family: bool = False         # OFF por default (muda métricas e pode “mexer no layout”)
#     scale_fonts: bool = False             # OFF por default
#     scale_strokes: bool = False           # OFF por default

#     # Panels
#     panel_min_width: float = 30.0         # mesma heurística do seu
#     round_panels: bool = True
#     recolor_panels: bool = True

#     verbose: bool = True


# # ==============================================================================
# # 3) Transformer final: baseado no seu, com Inkscape repair + gates
# # ==============================================================================
# class SVGTransformerFinal:
#     def __init__(self, theme: Theme, opt: Optional[Options] = None):
#         self.theme = theme
#         self.opt = opt or Options()
#         self.ns = {"svg": SVG_NS}

#         self.color_lookup: Dict[str, str] = {}
#         for cat, hex_codes in ORIGINAL_MAP.items():
#             if cat in theme.colors:
#                 for code in hex_codes:
#                     self.color_lookup[code.lower()] = theme.colors[cat]

#     # ---------- Inkscape repair: IDs duplicados ----------
#     def _dedupe_ids(self, root: etree._Element) -> int:
#         """
#         Renomeia IDs duplicados e atualiza referências url(#id) em:
#         - atributos (clip-path, mask, filter, marker-*)
#         - style=""
#         """
#         seen: Set[str] = set()
#         renamed = 0

#         # map old->new for later reference updates
#         # (quando tem duplicado, só os duplicados ganham suffix)
#         for el in root.iter():
#             if not isinstance(el.tag, str):
#                 continue
#             _id = el.get("id")
#             if not _id:
#                 continue
#             if _id not in seen:
#                 seen.add(_id)
#                 continue

#             # duplicate found → rename
#             i = 2
#             new_id = f"{_id}__dup{i}"
#             while new_id in seen:
#                 i += 1
#                 new_id = f"{_id}__dup{i}"
#             el.set("id", new_id)
#             seen.add(new_id)
#             renamed += 1

#             # Update references in entire doc: url(#old) -> url(#new) is tricky
#             # because we need to know which instance was intended.
#             # Best safe strategy: do NOT globally rewrite old->new, because old still exists.
#             # Instead, we only ensure uniqueness so Inkscape doesn't “pick the wrong one”.
#             # (Inkscape behavior with duplicate IDs is undefined; uniqueness already fixes most vanish issues.)

#         return renamed

#     # ---------- Inkscape repair: remove broken url(#id) ----------
#     def _collect_ids(self, root: etree._Element) -> Set[str]:
#         ids = set()
#         for el in root.iter():
#             if not isinstance(el.tag, str):
#                 continue
#             _id = el.get("id")
#             if _id:
#                 ids.add(_id)
#         return ids

#     def _drop_broken_url_refs(self, root: etree._Element) -> Tuple[int, int]:
#         """
#         Remove clip-path/mask/filter/marker-* that reference non-existent IDs.
#         This is the #1 cause of “sumir no Inkscape”.
#         """
#         ids = self._collect_ids(root)
#         attrs_removed = 0
#         style_removed = 0

#         url_keys = ["clip-path", "mask", "filter", "marker-start", "marker-mid", "marker-end"]

#         for el in root.iter():
#             if not isinstance(el.tag, str):
#                 continue

#             # attributes
#             for a in url_keys:
#                 v = el.get(a)
#                 if not v:
#                     continue
#                 m = URL_REF_RE.search(v)
#                 if m and m.group(1) not in ids:
#                     del el.attrib[a]
#                     attrs_removed += 1

#             # style props
#             style = el.get("style")
#             if style:
#                 d = parse_style(style)
#                 changed = False
#                 for k in list(d.keys()):
#                     if k not in url_keys:
#                         continue
#                     m = URL_REF_RE.search(d.get(k, ""))
#                     if m and m.group(1) not in ids:
#                         d.pop(k, None)
#                         style_removed += 1
#                         changed = True
#                 if changed:
#                     new_style = build_style(d)
#                     if new_style.strip():
#                         el.set("style", new_style)
#                     else:
#                         el.attrib.pop("style", None)

#         return attrs_removed, style_removed

#     # ---------- Shadow (optional, but disabled by default) ----------
#     def _ensure_defs(self, root: etree._Element) -> etree._Element:
#         defs = root.find("svg:defs", self.ns)
#         if defs is None:
#             defs = etree.Element(qn("defs"))
#             root.insert(0, defs)
#         return defs

#     def _inject_shadow_filter(self, root: etree._Element) -> bool:
#         if not self.opt.apply_shadow:
#             return False
#         opacity = float(self.theme.knobs.get("shadow_opacity", 0.0) or 0.0)
#         if opacity <= 0:
#             return False

#         defs = self._ensure_defs(root)

#         # remove existing shadow if any
#         for el in root.xpath("//*[@id='softShadow']"):
#             parent = el.getparent()
#             if parent is not None:
#                 parent.remove(el)

#         # Use classic chain (more consistent than feDropShadow)
#         f = etree.SubElement(defs, qn("filter"), id="softShadow",
#                              x="-25%", y="-25%", width="150%", height="150%",
#                              **{"color-interpolation-filters": "sRGB"})
#         etree.SubElement(f, qn("feGaussianBlur"), in_="SourceAlpha", stdDeviation="2.5", result="blur")
#         etree.SubElement(f, qn("feOffset"), in_="blur", dx="0", dy="1.5", result="off")
#         etree.SubElement(f, qn("feFlood"),
#                          **{"flood-color": "#000000", "flood-opacity": str(opacity)},
#                          result="flood")
#         etree.SubElement(f, qn("feComposite"), in_="flood", in2="off", operator="in", result="shadow")
#         merge = etree.SubElement(f, qn("feMerge"))
#         etree.SubElement(merge, qn("feMergeNode"), in_="shadow")
#         etree.SubElement(merge, qn("feMergeNode"), in_="SourceGraphic")
#         return True

#     # ---------- Your original color replacement ----------
#     def _update_style_string_colors(self, style_str: str) -> str:
#         if not style_str:
#             return ""
#         new_style = style_str
#         for old, new in self.color_lookup.items():
#             new_style = re.sub(re.escape(old), new, new_style, flags=re.IGNORECASE)
#         return new_style

#     # ---------- Main pass: minimal touch (no distortion) ----------
#     def apply_transformations(self, root: etree._Element) -> Dict[str, int]:
#         stats = {"colors": 0, "strokes": 0, "fonts": 0, "panels": 0}

#         stroke_scale = float(self.theme.knobs.get("stroke_scale", 1.0) or 1.0)
#         font_scale = float(self.theme.knobs.get("font_scale", 1.0) or 1.0)

#         for elem in root.iter():
#             if not isinstance(elem.tag, str):
#                 continue
#             tag = elem.tag.split("}")[-1]
#             style = elem.get("style") or ""

#             # --- A) Colors (attrs + style) ---
#             for attr in ["fill", "stroke"]:
#                 val = elem.get(attr)
#                 if val and val.lower() in self.color_lookup:
#                     elem.set(attr, self.color_lookup[val.lower()])
#                     stats["colors"] += 1

#             if style:
#                 new_style = self._update_style_string_colors(style)
#                 if new_style != style:
#                     elem.set("style", new_style)
#                     stats["colors"] += 1

#             # --- B) Stroke width ONLY if explicitly enabled AND scale != 1.0 ---
#             if self.opt.scale_strokes and stroke_scale != 1.0:
#                 sw = elem.get("stroke-width")
#                 if not sw and "stroke-width" in style:
#                     m = re.search(r"stroke-width\s*:\s*([^\s;]+)", style)
#                     sw = m.group(1) if m else None

#                 if sw:
#                     num, unit = parse_len(sw)
#                     if num is not None:
#                         elem.set("stroke-width", fmt_len(num * stroke_scale, unit, nd=3))
#                         stats["strokes"] += 1

#             # --- Panels (rect): same as your code, but shadow optional ---
#             if tag == "rect":
#                 w_num, _ = parse_len(elem.get("width", "0"))
#                 w_num = w_num or 0.0
#                 if w_num > self.opt.panel_min_width:
#                     if self.opt.round_panels:
#                         elem.set("rx", str(self.theme.knobs.get("corner_radius", 0)))
#                     if self.opt.apply_shadow and float(self.theme.knobs.get("shadow_opacity", 0) or 0) > 0:
#                         elem.set("filter", "url(#softShadow)")

#                     if self.opt.recolor_panels:
#                         fill = (elem.get("fill") or "").lower()
#                         if fill in ["#ffffff", "white", "none", ""]:
#                             elem.set("fill", self.theme.colors.get("bg_panel", fill))
#                     stats["panels"] += 1

#             # --- C) Typography: DO NOT TOUCH SIZE by default ---
#             if tag in ["text", "tspan"]:
#                 if self.opt.set_font_family:
#                     # Detect math-ish and preserve
#                     is_math = ("italic" in style.lower()) or ("times" in (elem.get("font-family") or "").lower())
#                     if is_math:
#                         elem.set("font-family", "'Times New Roman', serif")
#                         elem.set("font-style", "italic")
#                     else:
#                         # Add safe fallback for Inkscape
#                         ff = self.theme.font_family
#                         if "DejaVu Sans" not in ff:
#                             ff = f"{ff}, 'DejaVu Sans', Arial, sans-serif"
#                         elem.set("font-family", ff)
#                     stats["fonts"] += 1

#                 # scale font ONLY if explicitly enabled AND scale != 1.0
#                 if self.opt.scale_fonts and font_scale != 1.0:
#                     fs = elem.get("font-size")
#                     if fs:
#                         num, unit = parse_len(fs)
#                         if num is not None:
#                             # preserve original unit; DO NOT force px
#                             elem.set("font-size", fmt_len(num * font_scale, unit, nd=2))
#                             stats["fonts"] += 1

#         return stats

#     def transform(self, input_path: str, output_path: str) -> None:
#         if not os.path.exists(input_path):
#             raise FileNotFoundError(f"❌ Arquivo não encontrado: {input_path}")

#         parser = etree.XMLParser(remove_blank_text=True, recover=True)
#         tree = etree.parse(input_path, parser)
#         root = tree.getroot()

#         # 1) Inkscape repair passes (order matters)
#         renamed = 0
#         if self.opt.dedupe_duplicate_ids:
#             renamed = self._dedupe_ids(root)

#         attrs_removed = style_removed = 0
#         if self.opt.drop_broken_url_refs:
#             attrs_removed, style_removed = self._drop_broken_url_refs(root)

#         # 2) Shadow (optional)
#         shadow_ok = self._inject_shadow_filter(root)

#         # 3) Apply transformations (minimal touch)
#         stats = self.apply_transformations(root)

#         with open(output_path, "wb") as f:
#             tree.write(f, pretty_print=True, xml_declaration=True, encoding="utf-8")

#         if self.opt.verbose:
#             print(f"✅ Output: {output_path}")
#             print(f"   - IDs duplicados renomeados: {renamed}")
#             print(f"   - url(#id) quebradas removidas: attrs={attrs_removed}, style-props={style_removed}")
#             print(f"   - shadow: {'ON' if (self.opt.apply_shadow and shadow_ok) else 'OFF'}")
#             print(f"   - stats: {stats}")


# # ==============================================================================
# # 4) EXECUÇÃO PLUG AND PLAY
# # ==============================================================================
# if __name__ == "__main__":
#     SELECTED_THEME = "DARK_MODERN"

#     INPUT = "/home/gabriel/Downloads/Projeto_Fundamentação.svg"
#     OUTPUT = "/home/gabriel/Downloads/Projeto_Style.svg"

#     # SAFE DEFAULTS: sem distorção
#     OPT = Options(
#         dedupe_duplicate_ids=True,
#         drop_broken_url_refs=True,

#         apply_shadow=False,
#         set_font_family=False,   # <- mantenha False pra não mexer em métricas
#         scale_fonts=False,
#         scale_strokes=False,

#         panel_min_width=30.0,
#         round_panels=True,
#         recolor_panels=True,

#         verbose=True
#     )

#     styler = SVGTransformerFinal(THEMES_DATA[SELECTED_THEME], opt=OPT)
#     styler.transform(INPUT, OUTPUT)
